In [0]:
%run ./00_Config

In [0]:
from pyspark.sql.functions import current_timestamp

# Mapeamento dos caminhos de origem dos arquivos CSV na Landing Zone
data_sources = {
    "tb_base_atendentes": f"{LANDING_PATH}/base_atendentes.csv",
    "tb_base_motivos": f"{LANDING_PATH}/base_motivos.csv",
    "tb_canais": f"{LANDING_PATH}/canais.csv",
    "tb_chamados": f"{LANDING_PATH}/chamados.csv",
    "tb_chamados_hora": f"{LANDING_PATH}/Chamados_Hora.CSV",
    "tb_clientes": f"{LANDING_PATH}/clientes.csv",
    "tb_custos": f"{LANDING_PATH}/custos.csv",
    "tb_pesquisa_satisfacao": f"{LANDING_PATH}/pesquisa_satisfacao.csv"
}

print("Mapeamento de arquivos da camada bronze concluído.")

In [0]:
# Ingestão idempotente dos dados brutos com adição de metadados de auditoria
for table_name, file_path in data_sources.items():
    df_raw = spark.read.csv(file_path, header=True, inferSchema=True)
    
    df_raw \
        .withColumn("ingestion_timestamp", current_timestamp()) \
        .write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{BRONZE_SCHEMA}.{table_name}")

print("[SUCESSO] Ingestão da camada bronze finalizada de forma idempotente.")